# Data Aggregation & Cleaning

Goals:
- Aggregate PortWatch data to weekly data (Saturday to Friday, to match ACLED Data)
- Convert missing ACLED data to 0
- Merge PortWatch and ACLED datasets for analysis

## Read in Data
Read in Data generated in `01_data_collection.ipynb`

In [1]:
import os.path

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
import os

In [2]:
DATA_DIR_RAW = Path("data/raw")
DATA_DIR_PROCESSED = Path("data/processed")

acled = pd.read_parquet(Path.joinpath(DATA_DIR_PROCESSED, "acled_scope.parquet"))
portwatch = pd.read_csv(Path.joinpath(DATA_DIR_RAW, "portwatch_chokepoints.csv"))

print(f"ACLED data shape: {acled.shape}")
print(f"Portwatch data shape: {portwatch.shape}")

ACLED data shape: (8008, 14)
Portwatch data shape: (5492, 21)


## Aggregate PortWatch
Portwatch data is reported and published on a daily basis, while ACLED data is only published weekly. In order to merge datasets, we aggregate Portwatch into weekly data, where Saturday is the first day of the week.
- First, the `date` column is converted from `str` datatype to `datetime` datatype.
- Next, the week number is found for each day. If it is a Saturday or Sunday, it will be added to the next week.
    - Pandas includes a handy function `dt.to_period` which lets us start on a saturday


In [3]:
# convert to datetime format
portwatch['date'] = pd.to_datetime(portwatch['date'], format='ISO8601') #convert to pd.datetime format
portwatch['week_start'] = portwatch['date'].dt.to_period('W-FRI').dt.start_time #weekly period that goes to friday.

#drop date, day, objectid col, as these not needed anymore and interfere with the groupby function
portwatch = portwatch.drop(['date', 'month', 'year', 'day', 'ObjectId'], axis = "columns", errors='ignore')

Finally, we group the dataset by its week and chokepoint. We re-add the month and year column, but this time from the `week_start` column (instead of the original `date`), such that we can avoid issues where a single week spans over the end of a month or over the end of the year.

In [4]:
#groupby
portwatch['year'] = portwatch['week_start'].dt.year
portwatch['month'] = portwatch['week_start'].dt.month

portwatch_weekly = (
    portwatch
    .groupby(['week_start', 'portid', 'portname', 'year', 'month'], as_index=False)
    .sum(numeric_only=True)
)

Lastly, the ACLED dataset starts on the week of 2023-01-07, so we remove the first partial week from the `portwatch_weekly` dataset.

In [5]:
portwatch_weekly = portwatch_weekly[portwatch_weekly['week_start'] >= '2023-01-07'].copy()

# drop the incomplete final week
max_date = portwatch_weekly['week_start'].max()
portwatch_weekly = portwatch_weekly[portwatch_weekly['week_start'] < max_date].reset_index(drop=True)

# keep only weeks covered by the ACLED file, so later weeks are not counted as weeks without conflict
portwatch_weekly = portwatch_weekly[portwatch_weekly['week_start'] <= pd.to_datetime(acled['WEEK']).max()].reset_index(drop=True)

# all weeks in our date range, used to fill weeks without ACLED events
ALL_WEEKS = pd.date_range(portwatch_weekly['week_start'].min(), portwatch_weekly['week_start'].max(), freq='7D', name='WEEK')

# mandatory range check
assert (portwatch_weekly['n_total'] >= 0).all(), "Negative transits detected!"
assert (portwatch_weekly['capacity'] >= 0).all(), "Negative capacity detected!"

print(f"Clean PortWatch date range: {portwatch_weekly['week_start'].min().date()} to {portwatch_weekly['week_start'].max().date()}")
portwatch_weekly.head()

Clean PortWatch date range: 2023-01-07 to 2026-09-26


,week_start,portid,portname,year,month,n_container,n_dry_bulk,n_general_cargo,n_roro,n_tanker,n_cargo,n_total,capacity_container,capacity_dry_bulk,capacity_general_cargo,capacity_roro,capacity_tanker,capacity_cargo,capacity
0,2023-01-07,chokepoint1,Suez Canal,2023,1,110,134,53,20,180,317,497,5278362,7675700,208821,71425,8783294,13234320,22017616
1,2023-01-07,chokepoint4,Bab el-Mandeb Strait,2023,1,110,143,35,24,155,312,467,5721333,7572575,176131,42754,7791736,13512805,21304545
2,2023-01-07,chokepoint6,Strait of Hormuz,2023,1,84,88,23,8,296,203,499,2420164,2567562,100026,18722,14272826,5106482,19379313
3,2023-01-07,chokepoint7,Cape of Good Hope,2023,1,49,181,14,5,68,249,317,1889132,12009885,207375,14000,5520769,14120399,19641169
4,2023-01-14,chokepoint1,Suez Canal,2023,1,120,130,46,19,175,315,490,6301202,6831129,242091,57390,9603387,13431822,23035210


## Outliers
We flag weekly transits outside 1.5 times the interquartile range (IQR), separately per chokepoint.

Flagged weeks are kept, because they are real events and not data errors: they show traffic collapsing during the conflict, which is what we want to study. The new column `is_outlier` marks them.


In [6]:
q1 = portwatch_weekly.groupby('portid')['n_total'].transform('quantile', 0.25)
q3 = portwatch_weekly.groupby('portid')['n_total'].transform('quantile', 0.75)
iqr = q3 - q1
portwatch_weekly['is_outlier'] = (portwatch_weekly['n_total'] < q1 - 1.5 * iqr) | (portwatch_weekly['n_total'] > q3 + 1.5 * iqr)

portwatch_weekly[portwatch_weekly['is_outlier']].groupby('portname').agg(
    weeks=('week_start', 'count'),
    first_week=('week_start', 'min'),
    last_week=('week_start', 'max'),
    highest_transits=('n_total', 'max'),
)


,weeks,first_week,last_week,highest_transits
portname,,,,
Strait of Hormuz,29,2026-02-28,2026-09-26,127


## Filtering ACLED Data

To eliminate non-maritime ground combat noise (e.g. territorial shifts), records will be filtered specifically for the following types:
- event types: (already filtered)
    - '_Explosions/Remote violence_',
    - '_Battles_',
    - '_Strategic developments_'
- Subevent types:
    - ‘_Air/drone strike_’,
    - _Shelling/artillery/missile attack_’,
     - ‘_Disrupted weapons use_’ (capturing naval missile interceptions).

In [7]:
sub_events = ['Air/drone strike', 'Shelling/artillery/missile attack', 'Disrupted weapons use']
acled = acled[acled["SUB_EVENT_TYPE"].isin(sub_events)]
acled['SUB_EVENT_TYPE'].value_counts()

SUB_EVENT_TYPE
Air/drone strike                     1105
Shelling/artillery/missile attack     827
Disrupted weapons use                 689
Name: count, dtype: int64

In [8]:
## nan counts
acled.isnull().sum()

WEEK                     0
REGION                   0
COUNTRY                  0
ADMIN1                   0
EVENT_TYPE               0
SUB_EVENT_TYPE           0
EVENTS                   0
FATALITIES               0
POPULATION_EXPOSURE    896
DISORDER_TYPE            0
ID                       0
CENTROID_LATITUDE        0
CENTROID_LONGITUDE       0
maritime_sub_event       0
dtype: int64

The column `POPULATION_EXPOSURE` has 896 fields which are listed as NaN.

In [ ]:
acled[acled['POPULATION_EXPOSURE'].isnull()][['COUNTRY' ,'ADMIN1']].value_counts()

When selecting NaN values per country and Administrative region, it becomes evident that the vast majority of NaN values are located in the Indian ocean. As such, it becomes a valid assumption that events with NaN data fields occured in a remote enough region that there was little to no public population exposed to the event. As a result, all NaN fields are replaced with 0.

In [10]:
# fill missing population exposure
acled['POPULATION_EXPOSURE'] = acled['POPULATION_EXPOSURE'].fillna(0)
print(f"Number of NaN values in POPULATION_EXPOSURE: {acled['POPULATION_EXPOSURE'].isnull().sum()}")

# convert ACLED WEEK string to datetime to enable merging with PortWatch
acled['WEEK'] = pd.to_datetime(acled['WEEK'])

# align date range with PortWatch
acled = acled[acled['WEEK'] <= portwatch_weekly['week_start'].max()].copy()

# mandatory range check
assert (acled['EVENTS'] >= 0).all(), "Negative events count detected!"
assert (acled['FATALITIES'] >= 0).all(), "Negative fatalities detected!"

print(f"Clean ACLED date range: {acled['WEEK'].min().date()} to {acled['WEEK'].max().date()}")

Number of NaN values in POPULATION_EXPOSURE: 0
Clean ACLED date range: 2023-01-07 to 2026-09-26


# Combining Datasets
The ACLED and PortWatch datasets will now get combined to answer each research question with the applicable data.

| Dataset | Research question | ACLED | PortWatch |
|---|---|---|---|
| Dataset 1 | 1 (Hormuz) | Iran, UAE, Oman | Strait of Hormuz |
| Dataset 2 | 2 (Cape rerouting) | Yemen, plus events at sea as a separate column | Suez Canal, Bab el-Mandeb, Cape of Good Hope |
| Dataset 3 | 3 (Vessel types) | All countries | Strait of Hormuz, Bab el-Mandeb |


## Dataset 1
Combines the following:

- From ACLED, data with the `country` field, aggregated on a weekly basis:
    - Iran
    - UAE
    - Oman

- From Portwatch weekly aggregated data,`portname` field:
    - Strait of Hormuz (`portid == chokepoint6`)

Events at sea (`Indian Ocean`) are left out, because they cannot be located near Hormuz and many of them are likely Red Sea attacks.


In [ ]:
# PortWatch Strait of Hormuz
portwatch_ds1 = portwatch_weekly[portwatch_weekly['portid'] == 'chokepoint6'].copy()

# ACLED Hormuz (Iran, UAE, Oman) aggregated by week
acled_ds1_weekly = acled[~acled['COUNTRY'].isin(['Yemen', 'Indian Ocean'])].groupby('WEEK', as_index=False).agg(
    conflict_events=('EVENTS', 'sum'),
    fatalities=('FATALITIES', 'sum')
)

# fill weeks without events before calculating rolling means
acled_ds1_weekly = acled_ds1_weekly.set_index('WEEK').reindex(ALL_WEEKS, fill_value=0).reset_index()

# feature enrichment: 4-week rolling conflict intensity
acled_ds1_weekly['conflict_events_rolling4w'] = acled_ds1_weekly['conflict_events'].rolling(4, min_periods=1).mean()
acled_ds1_weekly['fatalities_rolling4w'] = acled_ds1_weekly['fatalities'].rolling(4, min_periods=1).mean()

# merge PortWatch & ACLED
dataset_1 = pd.merge(
    portwatch_ds1,
    acled_ds1_weekly,
    left_on='week_start',
    right_on='WEEK',
    how='left'
).drop(columns=['WEEK'])

dataset_1[['conflict_events', 'fatalities', 'conflict_events_rolling4w', 'fatalities_rolling4w']] = (
    dataset_1[['conflict_events', 'fatalities', 'conflict_events_rolling4w', 'fatalities_rolling4w']].fillna(0)
)

# save
dir_ds1 = Path('data/processed/dataset_1')
dir_ds1.mkdir(parents=True, exist_ok=True)
dataset_1.to_parquet(dir_ds1 / 'dataset_1_hormuz.parquet')

print(f"Dataset 1 shape: {dataset_1.shape}")
dataset_1.head()

## Dataset 2
Combines the following:
- From ACLED, data with the `country` field, aggregated on a weekly basis:
    - Yemen
    - Indian Ocean, as a separate column of events at sea (`red_sea_events`)

- From Portwatch weekly aggregated data:
    - Suez Canal (`portid == chokepoint1`)
    - Cape of Good Hope (`portid == chokepoint7`)
    - Bab el-Mandeb Strait(`portid == chokepoint4`)

`red_sea_events` is our proxy for attacks on shipping. ACLED records all events at sea under one region, so some events, especially in 2026, may not be in the Red Sea.


In [ ]:
#portwatch_ds_2 = portwatch_weekly[portwatch_weekly['portid'] != 'chokepoint6'] #includes all but straight of hormuz

# PortWatch: Suez, Bab el-Mandeb, and Cape of Good Hope
ports_q2 = ['chokepoint1', 'chokepoint4', 'chokepoint7']
portwatch_ds2 = portwatch_weekly[portwatch_weekly['portid'].isin(ports_q2)].copy()

# pivot transit counts to wide format to calculate route shares
pw_ds2_pivoted = portwatch_ds2.pivot(index='week_start', columns='portname', values='n_total').fillna(0)

# feature enrichment: Cape Route Share 
pw_ds2_pivoted['total_corridor_transits'] = (
    pw_ds2_pivoted['Cape of Good Hope'] + 
    pw_ds2_pivoted['Suez Canal'] + 
    pw_ds2_pivoted['Bab el-Mandeb Strait']
)
pw_ds2_pivoted['cape_share'] = pw_ds2_pivoted['Cape of Good Hope'] / pw_ds2_pivoted['total_corridor_transits']

# ACLED: Yemen conflict metrics
acled_ds2_weekly = acled[acled['COUNTRY'] == 'Yemen'].groupby('WEEK', as_index=False).agg(
    yemen_conflict_events=('EVENTS', 'sum'),
    yemen_fatalities=('FATALITIES', 'sum')
)

# fill weeks without events before calculating rolling means
acled_ds2_weekly = acled_ds2_weekly.set_index('WEEK').reindex(ALL_WEEKS, fill_value=0).reset_index()

# proxy for attacks on shipping: ACLED events recorded at sea
red_sea = acled[acled['COUNTRY'] == 'Indian Ocean']
acled_ds2_weekly['red_sea_events'] = red_sea.groupby('WEEK')['EVENTS'].sum().reindex(ALL_WEEKS, fill_value=0).values
acled_ds2_weekly['red_sea_events_rolling4w'] = acled_ds2_weekly['red_sea_events'].rolling(4, min_periods=1).mean()
acled_ds2_weekly['yemen_events_rolling4w'] = acled_ds2_weekly['yemen_conflict_events'].rolling(4, min_periods=1).mean()

# merge PortWatch & ACLED
dataset_2 = pd.merge(
    pw_ds2_pivoted.reset_index(),
    acled_ds2_weekly,
    left_on='week_start',
    right_on='WEEK',
    how='left'
).drop(columns=['WEEK'])

dataset_2[['yemen_conflict_events', 'yemen_fatalities', 'red_sea_events', 'red_sea_events_rolling4w', 'yemen_events_rolling4w']] = (
    dataset_2[['yemen_conflict_events', 'yemen_fatalities', 'red_sea_events', 'red_sea_events_rolling4w', 'yemen_events_rolling4w']].fillna(0)
)

# save
dir_ds2 = Path('data/processed/dataset_2')
dir_ds2.mkdir(parents=True, exist_ok=True)
dataset_2.to_parquet(dir_ds2 / 'dataset_2_rerouting.parquet')

print(f"Dataset 2 shape: {dataset_2.shape}")
dataset_2.head()

## Dataset 3
Combines the following:
- From ACLED, aggregated on a weekly basis:
    - all countries, as regional totals
    - the events of dataset 1 for Hormuz and `red_sea_events` for Bab el-Mandeb, to label each strait's weeks

- From Portwatch weekly aggregated data:
    - Strait of Hormuz (`portid == chokepoint6`)
    - Bab el-Mandeb Strait (`portid == chokepoint4`)

A week counts as High conflict for a strait if it has more events than that strait's 75th percentile.


In [ ]:
# PortWatch: Strait of Hormuz and Bab el-Mandeb Strait
dataset_3 = portwatch_weekly[portwatch_weekly['portid'].isin(['chokepoint4', 'chokepoint6'])].copy()

# ACLED: Total regional conflict
acled_ds3_weekly = acled.groupby('WEEK', as_index=False).agg(
    total_regional_events=('EVENTS', 'sum'),
    total_regional_fatalities=('FATALITIES', 'sum')
)
acled_ds3_weekly['regional_events_rolling4w'] = acled_ds3_weekly['total_regional_events'].rolling(4, min_periods=1).mean()


# merge PortWatch & ACLED
dataset_3 = pd.merge(
    dataset_3,
    acled_ds3_weekly,
    left_on='week_start',
    right_on='WEEK',
    how='left'
).drop(columns=['WEEK'])

dataset_3['total_regional_events'] = dataset_3['total_regional_events'].fillna(0)
dataset_3['total_regional_fatalities'] = dataset_3['total_regional_fatalities'].fillna(0)
# feature enrichment 1: high vs. low conflict weeks, per strait
strait_events = pd.concat([
    acled_ds1_weekly.set_index('WEEK')['conflict_events'].rename('chokepoint6'),
    acled_ds2_weekly.set_index('WEEK')['red_sea_events'].rename('chokepoint4'),
], axis=1).rename_axis('week_start').reset_index().melt('week_start', var_name='portid', value_name='strait_events')
strait_events['conflict_intensity_level'] = np.where(
    strait_events['strait_events'] > strait_events.groupby('portid')['strait_events'].transform('quantile', 0.75), 'High', 'Low'
)
dataset_3 = dataset_3.merge(strait_events, on=['week_start', 'portid'], how='left')

# feature enrichment 2: pre-crisis baseline retention index (Jan 2023 - Sep 2023)
baseline_mask = (dataset_3['week_start'] >= '2023-01-07') & (dataset_3['week_start'] < '2023-10-01')
baselines = dataset_3[baseline_mask].groupby('portid')[['n_container', 'n_tanker', 'n_dry_bulk']].mean()

# calculate retention ratios
dataset_3 = dataset_3.merge(baselines, on='portid', suffixes=('', '_baseline'))
dataset_3['retention_container'] = dataset_3['n_container'] / dataset_3['n_container_baseline']
dataset_3['retention_tanker'] = dataset_3['n_tanker'] / dataset_3['n_tanker_baseline']
dataset_3['retention_dry_bulk'] = dataset_3['n_dry_bulk'] / dataset_3['n_dry_bulk_baseline']

# save
dir_ds3 = Path('data/processed/dataset_3')
dir_ds3.mkdir(parents=True, exist_ok=True)
dataset_3.to_parquet(dir_ds3 / 'dataset_3_vessel_resilience.parquet')

print(f"Dataset 3 shape: {dataset_3.shape}")
dataset_3.head()

### Verification of the output Parquet Files 

In [14]:
import pandas as pd
from pathlib import Path

ds1 = pd.read_parquet("data/processed/dataset_1/dataset_1_hormuz.parquet")
ds2 = pd.read_parquet("data/processed/dataset_2/dataset_2_rerouting.parquet")
ds3 = pd.read_parquet("data/processed/dataset_3/dataset_3_vessel_resilience.parquet")

print("DS1 (Hormuz):", ds1.shape)
print("DS2 (Cape Rerouting):", ds2.shape)
print("DS3 (Vessel Resilience):", ds3.shape)

DS1 (Hormuz): (195, 24)
DS2 (Cape Rerouting): (195, 11)
DS3 (Vessel Resilience): (390, 31)
